In [21]:
from src.sampling.main import stratified_spatial_kfold_dual #Dont know why but this has to be initialised first else kernel crashes

import torch
import os
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from datetime import datetime
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader as GeometricDataLoader
from torch_geometric.transforms import ToUndirected

from src.performance_logger import PerformanceLogger
from models.gnn import GNNInductiveHetero
from src.utils import read_config
from src.raingauge.utils import (
  load_raingauge_dataset
)
from src.radar.utils import load_radar_dataset, load_processed_dataset
from training.logic_hetero import train_epoch, validate, test_model
from src.graph.radargraph import RadarGraph
from src.graph.gaugegraphnew import GaugeGraphNew, HeterogeneousWeatherGraphDatasetInductive
from src.cml.utils import load_cml_dataset

import networkx as nx
from sklearn.neighbors import NearestNeighbors

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = read_config('config.yaml')
batch_size = config['training_params']['batch_size']
fold_count = config['training_params']['fold_count']

In [ ]:
experiment_name = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_new"
os.makedirs(f"experiments/{experiment_name}", exist_ok=True)
perf = PerformanceLogger(f"experiments/{experiment_name}/training_log.jsonl")

In [ ]:
uptime_threshold = config['filters']['uptime_threshold']
start_year = config['dataset_parameters']['start_year']
end_year = config['dataset_parameters']['end_year']
raingauge_df, raingauge_station_mappings_df = load_raingauge_dataset(start = start_year, end = end_year, uptime_threshold=uptime_threshold)

In [ ]:
raingauge_df

In [ ]:
raingauge_station_mappings_df

In [15]:
x_coords = np.arange(103.605, 104.1, 0.01)
y_coords = np.arange(1.185,1.5, 0.01)
print(len(x_coords))
print(len(y_coords))

dummy_nodes = []
count = 0
for x in x_coords:
  for y in y_coords:
    id = f"Dummy{count}"
    node = {"id": id, "latitude": y, "longitude": x, "name": id}
    count += 1
    dummy_nodes.append(node)

dummy_df = pd.DataFrame(dummy_nodes)

50
32


In [16]:
print(dummy_df)

             id  latitude  longitude       name
0        Dummy0     1.185    103.605     Dummy0
1        Dummy1     1.195    103.605     Dummy1
2        Dummy2     1.205    103.605     Dummy2
3        Dummy3     1.215    103.605     Dummy3
4        Dummy4     1.225    103.605     Dummy4
...         ...       ...        ...        ...
1595  Dummy1595     1.455    104.095  Dummy1595
1596  Dummy1596     1.465    104.095  Dummy1596
1597  Dummy1597     1.475    104.095  Dummy1597
1598  Dummy1598     1.485    104.095  Dummy1598
1599  Dummy1599     1.495    104.095  Dummy1599

[1600 rows x 4 columns]


In [29]:
G = nx.Graph()
coords = dummy_df[['longitude', 'latitude']].values

ball_tree = NearestNeighbors(n_neighbors=8, algorithm='ball_tree').fit(coords)

distances, indices = ball_tree.kneighbors(coords)

for idx, row in dummy_df.iterrows():
    G.add_node(idx, lat=row['latitude'], lon=row['longitude'])

for i, neighbors in enumerate(indices):
    for j, neighbor_idx in enumerate(neighbors[1:]):
      dist = distances[i][j + 1]

      G.add_edge(i, neighbor_idx, weight=dist)


In [30]:
edge_arr = np.array(G.edges)

In [31]:
edge_arr

array([[   0,    1],
       [   0,   32],
       [   0,   33],
       ...,
       [1597, 1598],
       [1597, 1599],
       [1598, 1599]], shape=(6318, 2))

In [18]:
test_df = pd.concat([raingauge_station_mappings_df, dummy_df], axis = 0)

In [ ]:
#Create dummy locations
#The dummy locations will act as "erroneous sensors that contain nan values so that the model will create a mask for them"
dummy_locations = 

In [ ]:
radar_df = load_processed_dataset(folder_name='database/processed_radar_dataset.pkl')
cml_df, cml_coordinates_df = load_cml_dataset(config['dataset_parameters']['cml_folder'])
cml_df.fillna(0)

In [ ]:
#MERGE THE RADAR AND GAUGE DATA

radar_max = radar_df["timestamp"].max()
radar_min = radar_df["timestamp"].min()

raingauge_max = raingauge_df.index.max()
raingauge_min = raingauge_df.index.min()

cml_max = cml_df['timestamp'].max()
cml_min = cml_df['timestamp'].min()

radar_cols = radar_df.columns
raingauge_cols = raingauge_df.columns
cml_cols = cml_df.columns

merged_df = pd.DataFrame()
merged_df['timestamp']= pd.date_range(start=min(radar_min, raingauge_min, cml_min), end = (max(radar_max, raingauge_max, cml_max)), freq='15min')
merged_df = merged_df.set_index('timestamp')
cml_df = cml_df.set_index('timestamp')
radar_df = radar_df.set_index('timestamp')
print(merged_df)

merged_df = merged_df.join(radar_df, how='left')
merged_df = merged_df.join(raingauge_df, how='left')
merged_df = merged_df.join(cml_df, how='left')

In [ ]:
merged_df.sort_values(by='timestamp')

In [ ]:
merged_df
radar_df = merged_df[radar_cols]
raingauge_df = merged_df[raingauge_cols]

In [ ]:
raingauge_df

In [ ]:
radar_df

In [ ]:
rainfall_values = torch.tensor(raingauge_df.values.T)
rainfall_validity = torch.tensor(raingauge_df.notna().astype(int).values.T)
raingauge_features = torch.stack([rainfall_values, rainfall_validity], dim = 2)
raingauge_features.shape

In [ ]:
gauge_graph_arr = []
for i in range(1):
  gauge_graph = GaugeGraphNew(raingauge_df, raingauge_station_mappings_df, split_info = split_info, knn=5)
  radar_graph = RadarGraph(radar_df)
  gauge_graph.add_heterodata(snapshots=radar_graph, layer_name='radar', knn=9)
  gauge_graph_arr.append(gauge_graph)
  

In [ ]:
model=torch.load("testing/gnn_model.pth", map_location=device)

In [ ]:
train_loader_arr = []
val_loader_arr = []
test_loader_arr = []
for i in range(fold_count):
    train_loader = GeometricDataLoader(
    HeterogeneousWeatherGraphDatasetInductive(gauge_graph_arr[i].get_train_heterodata()), #Need to convert to timestep wise data
    batch_size=batch_size,
    shuffle= False,
    )

    val_loader = GeometricDataLoader(
    HeterogeneousWeatherGraphDatasetInductive(gauge_graph_arr[i].get_validation_heterodata()),
    batch_size=batch_size,
    shuffle=False,
    )

    test_loader = GeometricDataLoader(
    HeterogeneousWeatherGraphDatasetInductive(gauge_graph_arr[i].get_test_heterodata()),
    batch_size = batch_size, 
    shuffle = False
    )

    train_loader_arr.append(train_loader)
    val_loader_arr.append(val_loader)
    test_loader_arr.append(test_loader)

In [ ]:
print(next(iter(test_loader_arr[0])))

In [ ]:
RMSE = test_model(model, raingauge_station_mappings_df, test_loader, device, fold = 0, experiment_name=experiment_name)